# PUMP Subset Selection with Deployment Data

Adapted from the supplied FAN LR/SVC/DT notebook, preserving its section order, pairwise feature selection, pairwise `RobustScaler`, model settings, voting, and evaluation schedule.

Use the four CSVs produced by the PUMP extraction notebooks. Each row is one `fault_id@pump_id@sample_id`; Air/Water and region identifiers are already present in feature names. Both states and every region therefore stay together during splitting. No `config_inference.json` is needed.

Feature/model selection uses at most ten balanced folds. Final evaluation uses the complete least-common-multiple schedule of per-fault pump combinations, with approximately 30% of each fault's pumps in testing (four when more than ten are available). Groups are identified by **fault ID + pump number**, matching the supplied data's per-fault pump numbering; this assumes the same number under different faults does not identify the same physical pump.

Necessary adaptations: four PUMP inputs, pump metadata excluded from predictors, no FAN-only feature exclusions/RPM files, automatic healthy/single-fault IDs and defect-pattern labels, PUMP export names, audit fold 0, and correction of the source's `max_feats....` typo. The supplied extracted dataset is expected to contain 18 classes including healthy, giving 153 pairs; neither number is hardcoded.

As in the FAN reference, selected columns/model choices are reused in the final evaluation. These are development scores, not an independent estimate on unseen production pumps; that would require an untouched test set or nested selection. Per-fold scalers still fit only training rows.

In [ ]:
# Optional Google Colab setup; local Jupyter skips the mount.
try:
    from google.colab import drive
except ImportError:
    print("Local Jupyter: Google Drive mount skipped.")
else:
    drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Edit FEATURE_DIR if the four extracted CSVs are elsewhere.
# In Colab, use the full folder path under /content/drive/MyDrive/...
FEATURE_DIR = Path("pump_features")
OUTPUT_DIR = Path("pump_subset_selection_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Setup and imports

Configure thread limits, import dependencies, and set warning/display behavior.

In [ ]:
import os

# Limit native-library thread usage for reproducible, resource-conscious runs.
# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["MKL_NUM_THREADS"] = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = "1"
# os.environ["NUMEXPR_NUM_THREADS"] = "1"
# os.environ["POLARS_MAX_THREADS"] = "1"

import copy
import gc
import json
import warnings
import zlib
from itertools import combinations
from math import lcm
from statistics import mode

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from tqdm import tqdm

np.set_printoptions(suppress=True)

with warnings.catch_warnings():
    warnings.filterwarnings(action="ignore", category=ConvergenceWarning)

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

## 2. Load, validate, and merge feature data

Load the feature tables, normalize sample keys, validate uniqueness, merge one-to-one, and build metadata columns.

In [ ]:
feature_paths = {
    'analog': FEATURE_DIR / 'PUMP_ctvt_singlefault_peaks_freq.csv',
    'digital': FEATURE_DIR / 'PUMP_digital_feats_updt.csv',
    'physics': FEATURE_DIR / 'PUMP_physics_features.csv',
    'stats': FEATURE_DIR / 'PUMP_stats_features_2.csv',
}

# These are identifiers/descriptors, never input features.
metadata_cols = [
    'sample_id_key', 'fault_id', 'pump_id', 'sample_id', 'pump_id_key',
    'fault_type', 'defect_id', 'id', 'pump_no', 'state', 'state_id_key',
    'source_namespace', 'key', 'id_', 'sample_no', 'sno',
]
feature_files = {}
for name, path in feature_paths.items():
    df = pd.read_csv(path, dtype={col: str for col in metadata_cols})
    if df.empty or 'sample_id_key' not in df.columns:
        raise ValueError(f'{name}: empty file or sample_id_key missing')
    if df['sample_id_key'].isna().any():
        raise ValueError(f'{name}: missing sample keys')
    df['sample_id_key'] = df['sample_id_key'].str.strip()
    parts = df['sample_id_key'].str.split('@', expand=True)
    if parts.shape[1] != 3 or parts.isna().any().any():
        raise ValueError(f'{name}: expected fault_id@pump_id@sample_id keys')
    if not parts[0].str.fullmatch(r'[0-9]+_').all():
        raise ValueError(f'{name}: malformed fault IDs')
    for position in [1, 2]:
        numbers = pd.to_numeric(parts[position], errors='raise')
        if not np.isfinite(numbers).all() or (numbers % 1 != 0).any() or (numbers < 0).any():
            raise ValueError(f'{name}: pump/sample IDs must be nonnegative integers')
        parts[position] = numbers.astype('int64').astype(str)
    df['sample_id_key'] = parts.agg('@'.join, axis=1)
    for position, col in enumerate(['fault_id', 'pump_id', 'sample_id']):
        if col in df:
            values = df[col].str.strip()
            if position:
                values = pd.to_numeric(values, errors='raise')
                matches = values.eq(pd.to_numeric(parts[position]))
            else:
                matches = values.eq(parts[position])
            if not matches.all():
                raise ValueError(f'{name}: {col} disagrees with sample_id_key')
    if df['sample_id_key'].duplicated().any():
        raise ValueError(f'{name}: duplicate sample_id_key after normalization')
    # Avoid merging four copies of identifiers with _x/_y suffixes.
    df = df.drop(columns=[col for col in metadata_cols if col != 'sample_id_key'], errors='ignore')
    feature_cols = [col for col in df if col != 'sample_id_key']
    if not feature_cols:
        raise ValueError(f'{name}: no feature columns')
    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='raise')
    if not np.isfinite(df[feature_cols].to_numpy(dtype=float)).all():
        raise ValueError(f'{name}: NaN/Inf values in features')
    feature_files[name] = df
    print(name, df.shape, 'unique samples:', df['sample_id_key'].nunique())

reference_keys = set(feature_files['analog']['sample_id_key'])
input_coverage_rows = []
for name, df in feature_files.items():
    keys = set(df['sample_id_key'])
    missing, extra = reference_keys - keys, keys - reference_keys
    input_coverage_rows.append({'file': name, 'samples': len(keys),
                               'missing_vs_analog': len(missing), 'extra_vs_analog': len(extra)})
    if missing or extra:
        raise ValueError(f'{name}: sample keys differ from analog. '
                         f'Missing: {sorted(missing)[:10]}; extra: {sorted(extra)[:10]}')
input_coverage = pd.DataFrame(input_coverage_rows)

merged = feature_files['digital'].copy()
for name in ['analog', 'physics', 'stats']:
    df = feature_files[name]
    overlap = (set(merged) & set(df)) - {'sample_id_key'}
    if overlap:
        raise ValueError(f'{name}: duplicate feature names: {sorted(overlap)}')
    merged = pd.merge(merged, df, on='sample_id_key', how='inner', validate='one_to_one')

merged['pump_no'] = merged['sample_id_key'].str.split('@').str[1].astype(int)
merged['id'] = merged['sample_id_key'].str.split('@').str[0]
merged = merged.reset_index(drop=True)
drop_cols = ['id', 'pump_no', 'sample_id_key']
print('Merged feature count:', len(merged.columns) - len(drop_cols))
merged

## 3. Feature pruning and model-input preparation

Keep PUMP state/region features. The FAN-only deployment exclusions and gyroscope-band exclusions do not apply automatically to these PUMP columns. Constant-feature removal remains inside training-fold ANOVA, as in the reference.

In [ ]:
merged = merged.reset_index(drop=True)
# Add exact PUMP feature names here only if an exclusion is required.
cols_to_drop_for_deployment = []
merged = merged.drop(columns=cols_to_drop_for_deployment)
merged

In [ ]:
merged1 = merged
del feature_files
gc.collect()
merged1

## 4. Dataset inspection

Quick checks for sample identifiers and the final feature list.

In [ ]:
merged1['sample_id_key']

In [ ]:
print(','.join(merged1.columns))

## 5. Pump-combination split utilities

Select approximately 30% of the physical pumps for testing within each fault and cap testing at four pumps when more than ten are available. A deterministic balanced subset is used during feature/model selection; the final combined evaluation uses the complete least-common-multiple schedule and covers every per-fault combination equally.

In [ ]:
def get_test_pump_count(n_pumps, test_fraction=0.30):
    '''Return the nearest integer to 30% of pumps, with the requested cap.'''
    n_pumps = int(n_pumps)
    if n_pumps < 2:
        raise ValueError(
            f"At least two physical pumps are required; found {n_pumps}."
        )

    if n_pumps > 10:
        return 4

    # Round half upward instead of using Python's round-to-even behavior.
    n_test_pumps = int(np.floor((n_pumps * test_fraction) + 0.5))
    return min(max(1, n_test_pumps), n_pumps - 1)


def select_balanced_combinations(
    fault_combinations,
    fault_pumps,
    max_folds,
    rng
):
    '''Select deterministic combinations while balancing test exposure per pump.'''
    shuffled = [
        fault_combinations[index]
        for index in rng.permutation(len(fault_combinations))
    ]

    if max_folds is None or len(shuffled) <= int(max_folds):
        return shuffled
    if int(max_folds) < 1:
        raise ValueError("max_folds must be at least 1 or None.")

    remaining = shuffled.copy()
    selected = []
    exposure = {pump_id: 0 for pump_id in fault_pumps}

    while remaining and len(selected) < int(max_folds):
        def balance_score(combination):
            next_counts = [
                exposure[pump_id] + int(pump_id in combination)
                for pump_id in fault_pumps
            ]
            return (
                max(next_counts) - min(next_counts),
                sum(count ** 2 for count in next_counts),
                sum(exposure[pump_id] for pump_id in combination)
            )

        best_index = min(
            range(len(remaining)),
            key=lambda index: balance_score(remaining[index])
        )
        chosen = remaining.pop(best_index)
        selected.append(chosen)

        for pump_id in chosen:
            exposure[pump_id] += 1

    return selected


def build_pump_combination_plan(df, random_state=42, max_folds=None):
    '''
    Build deterministic test-pump combinations independently for every fault.

    Each physical group is identified by fault_id@pump_id. With max_folds=None,
    every test-pump combination is retained and the least-common-multiple
    schedule gives every combination equal exposure. With max_folds set, at
    most that many deterministic combinations are retained per fault, chosen
    to keep individual pump exposure as balanced as possible.
    '''
    if df.empty:
        raise ValueError("Cannot split an empty DataFrame.")

    required_cols = {"id", "pump_no"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

    split_plan = {}

    for fault_id in sorted(df["id"].unique()):
        fault_pumps = tuple(sorted(
            df.loc[df["id"] == fault_id, "pump_no"]
              .astype(int)
              .unique()
              .tolist()
        ))

        n_test_pumps = get_test_pump_count(len(fault_pumps))
        fault_combinations = list(combinations(fault_pumps, n_test_pumps))

        # The seed depends on the fault ID, not its position in the class list.
        # Common faults retain the same order when the selected class list changes.
        fault_seed = (
            int(random_state) + zlib.crc32(str(fault_id).encode("utf-8"))
        ) % (2 ** 32)
        rng = np.random.default_rng(fault_seed)
        split_plan[fault_id] = select_balanced_combinations(
            fault_combinations=fault_combinations,
            fault_pumps=fault_pumps,
            max_folds=max_folds,
            rng=rng
        )

    combination_counts = [
        len(fault_combinations)
        for fault_combinations in split_plan.values()
    ]
    if max_folds is None:
        n_folds = lcm(*combination_counts)
    else:
        n_folds = max(combination_counts)

    return split_plan, n_folds


def train_test_split_2(df, fold=0, split_plan=None, n_folds=None, random_state=42):
    '''Create one leakage-free split from the exhaustive combination plan.'''
    df = df.reset_index(drop=True)

    if split_plan is None or n_folds is None:
        split_plan, n_folds = build_pump_combination_plan(
            df,
            random_state=random_state
        )

    if not 0 <= int(fold) < int(n_folds):
        raise IndexError(
            f"Fold {fold} is invalid. Available folds: {n_folds}."
        )

    expected_faults = set(df["id"].unique())
    if set(split_plan) != expected_faults:
        raise ValueError(
            "The split plan does not match the fault IDs in the DataFrame."
        )

    test_mask = pd.Series(False, index=df.index)

    for fault_id, fault_combinations in split_plan.items():
        test_pumps = fault_combinations[fold % len(fault_combinations)]
        test_mask |= (
            (df["id"] == fault_id) &
            df["pump_no"].astype(int).isin(test_pumps)
        )

    test_data = df.loc[test_mask].reset_index(drop=True)
    train_data = df.loc[~test_mask].reset_index(drop=True)

    train_groups = set(zip(train_data["id"], train_data["pump_no"].astype(int)))
    test_groups = set(zip(test_data["id"], test_data["pump_no"].astype(int)))
    overlapping_groups = train_groups & test_groups
    if overlapping_groups:
        raise ValueError(
            "Physical-pump leakage detected.\n"
            f"Overlapping groups: {sorted(overlapping_groups)}"
        )

    train_faults = set(train_data["id"])
    test_faults = set(test_data["id"])
    if train_faults != expected_faults:
        raise ValueError(
            "Training does not contain every fault class.\n"
            f"Expected: {sorted(expected_faults)}\n"
            f"Found: {sorted(train_faults)}"
        )
    if test_faults != expected_faults:
        raise ValueError(
            "Testing does not contain every fault class.\n"
            f"Expected: {sorted(expected_faults)}\n"
            f"Found: {sorted(test_faults)}"
        )

    return train_data, test_data


def anova(X, y, no_of_cols=1000):
    valid_cols = [
        col for col in X.columns
        if X[col].nunique(dropna=False) > 1
    ]
    X = X[valid_cols]

    if X.shape[1] == 0:
        raise ValueError("No non-constant feature columns are available.")

    k = min(int(no_of_cols), X.shape[1])
    fs = SelectKBest(score_func=f_classif, k=k)
    fit = fs.fit(X, y)

    feature_score = pd.DataFrame({
        "Input_Features": X.columns,
        "F_Score": fit.scores_,
        "pvals": fit.pvalues_
    })
    feature_score = feature_score.replace([np.inf, -np.inf], np.nan)
    feature_score = feature_score.dropna(subset=["F_Score"])

    return feature_score.sort_values(
        by="F_Score",
        ascending=False
    ).reset_index(drop=True)

## 6. Model transformers and registry

Define the per-combination estimator wrapper and the base classifier configurations.

In [ ]:
class LRTransformer(BaseEstimator, TransformerMixin):

    def __init__(self, model=None, ids='', columns=None):
        self.model = model
        self.ids = ids
        self.columns = columns

    def fit(self, X, y=None):
        list_ids = self.ids.split('@')
        X = pd.DataFrame(X, columns=self.columns)
        X = X[X['id'].isin(list_ids)].reset_index(drop=True)
        y = X['id']
        X = X.drop(drop_cols, axis=1)
        self.scaler_ = RobustScaler()
        X = self.scaler_.fit_transform(X)
        self.model_ = copy.deepcopy(self.model)
        self.model_.fit(X, y)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns)
        X = X.drop(drop_cols, axis=1)
        X = self.scaler_.transform(X)
        return self.model_.predict(X).reshape(-1, 1)

In [ ]:
MODEL_META = {
    'lr': LogisticRegression(solver='liblinear', class_weight='balanced', max_iter=1000, random_state=42),
    'svc': SVC(class_weight='balanced', cache_size=500),
    'dt': DecisionTreeClassifier(class_weight='balanced', max_depth=5, min_samples_leaf=2, random_state=42)
    }

## 7. Feature-selection and evaluation helpers

Use deterministic pump-balanced folds for feature selection and preliminary per-model evaluation. The complete exhaustive schedule is reserved for the final selected-model ensemble.

In [ ]:
def feature_selection(
    df,
    key,
    model=None,
    n_feats=100,
    random_state=42,
    max_folds=10
):
    if model is None:
        model = LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )

    subset_df = df[df["id"].isin(key.split("@"))].reset_index(drop=True)
    feature_cols = [
        col for col in subset_df.columns
        if col not in drop_cols
    ]

    max_feats = min(int(n_feats), len(feature_cols))
    if max_feats < 1:
        raise ValueError(f"No feature columns are available for {key}.")

    candidate_ks = list(range(10, max_feats + 1, 10))
    if max_feats not in candidate_ks:
        candidate_ks.append(max_feats)
    candidate_ks = sorted(set(candidate_ks))

    scores_by_k = {k: [] for k in candidate_ks}
    split_plan, n_folds = build_pump_combination_plan(
        subset_df,
        random_state=random_state,
        max_folds=max_folds
    )

    for fold in range(n_folds):
        train_data, test_data = train_test_split_2(
            subset_df,
            fold=fold,
            split_plan=split_plan,
            n_folds=n_folds
        )

        feature_score = anova(
            train_data[feature_cols],
            train_data["id"],
            max_feats
        )
        ranked_cols = feature_score["Input_Features"].tolist()
        if not ranked_cols:
            raise ValueError(f"No usable features for {key} in fold {fold}.")

        scaler = RobustScaler()
        X_train_scaled = pd.DataFrame(
            scaler.fit_transform(train_data[ranked_cols]),
            columns=ranked_cols
        )
        X_test_scaled = pd.DataFrame(
            scaler.transform(test_data[ranked_cols]),
            columns=ranked_cols
        )

        for k in candidate_ks:
            cols = ranked_cols[:k]
            iter_model = copy.deepcopy(model)
            iter_model.fit(X_train_scaled[cols], train_data["id"])
            preds = iter_model.predict(X_test_scaled[cols])
            score = f1_score(
                test_data["id"],
                preds,
                average="macro",
                zero_division=0
            )
            scores_by_k[k].append(score)

        del train_data, test_data
        del feature_score, ranked_cols
        del scaler, X_train_scaled, X_test_scaled
        del iter_model, preds

    mean_scores = {
        k: float(np.mean(scores))
        for k, scores in scores_by_k.items()
    }
    best_k = max(mean_scores, key=mean_scores.get)

    final_feature_score = anova(
        subset_df[feature_cols],
        subset_df["id"],
        max_feats
    )
    final_cols = final_feature_score["Input_Features"].tolist()[:best_k]

    return mean_scores[best_k], final_cols


def algo_test(
    df,
    model=None,
    cols_meta=None,
    random_state=42,
    max_folds=10
):
    print(model)

    true_labels = []
    pred_labels = []

    le = LabelEncoder()
    _ = le.fit_transform(df["id"])
    mapped_classes = {
        i: class_id for i, class_id in enumerate(le.classes_)
    }

    combined_process = FeatureUnion([
        (
            f"pipe_{i}",
            Pipeline([
                (
                    f"{key}ct",
                    ColumnTransformer(
                        transformers=[
                            ("cols", "passthrough", cols + drop_cols)
                        ]
                    )
                ),
                (
                    f"{key}lr",
                    LRTransformer(
                        model=copy.deepcopy(model),
                        ids=key,
                        columns=cols + drop_cols
                    )
                )
            ])
        )
        for i, (key, cols) in enumerate(cols_meta.items())
    ])

    split_plan, n_folds = build_pump_combination_plan(
        df,
        random_state=random_state,
        max_folds=max_folds
    )
    print("Balanced preliminary folds:", n_folds)

    for fold in tqdm(range(n_folds), desc="Preliminary evaluation folds"):
        train_data, test_data = train_test_split_2(
            df,
            fold=fold,
            split_plan=split_plan,
            n_folds=n_folds
        )

        iter_combined_process = copy.deepcopy(combined_process)
        iter_combined_process.fit(train_data)
        output = iter_combined_process.transform(test_data)

        pred_labels.extend([mode(arr) for arr in output])
        true_labels.extend(test_data["id"].tolist())

        del iter_combined_process, output
        del train_data, test_data

    gc.collect()

    plt.figure(figsize=(28, 18))
    conf_matr = confusion_matrix(true_labels, pred_labels)
    arr_true_labels = np.array(true_labels)
    sample_weights = [
        arr_true_labels[arr_true_labels == class_id].shape[0]
        for class_id in sorted(np.unique(arr_true_labels))
    ]
    normalized_conf_matr = np.round(
        np.array([
            row / sample_weights[i]
            for i, row in enumerate(conf_matr)
        ]),
        3
    )
    sns.heatmap(normalized_conf_matr, annot=True)

    print(mapped_classes)
    print(classification_report(true_labels, pred_labels, zero_division=0))
    print(
        "Macro F1:",
        f1_score(true_labels, pred_labels, average="macro", zero_division=0)
    )
    print(
        "Weighted F1:",
        f1_score(true_labels, pred_labels, average="weighted", zero_division=0)
    )

    return true_labels, pred_labels

## 8. Fault subsets and class mapping

Select healthy plus every single-fault ID present in the extracted PUMP features. Remove IDs only through the explicit `remove_ids` list. Class and pair counts adapt automatically; the supplied default PUMP dataset has 18 classes and 153 pairs.

In [ ]:
single_ids = np.array([
    fault_id
    for fault_id in merged1['id'].unique()
    if sum(c != '0' for c in fault_id.rstrip('_')) <= 1
])
MAX_SELECTION_FOLDS = 10
remove_ids = []  # No FAN-specific exclusions.
filt_unique_ids = sorted(set(single_ids) - set(remove_ids))
if len(filt_unique_ids) < 2:
    raise ValueError('At least two healthy/single-fault classes are required')
single_id_subset = merged1[merged1['id'].isin(filt_unique_ids)].reset_index(drop=True)
all_combs = ['@'.join(comb) for comb in combinations(filt_unique_ids, 2)]

healthy_ids = [fault_id for fault_id in filt_unique_ids if set(fault_id.rstrip('_')) == {'0'}]
if len(healthy_ids) != 1:
    raise ValueError('Expected exactly one healthy fault ID')
healthy_label = healthy_ids[0]
print('Classes used (including healthy):', len(filt_unique_ids))
print('Pairwise classifiers:', len(all_combs))
print('Removed IDs:', remove_ids)
selection_plan, selection_n_folds = build_pump_combination_plan(
    single_id_subset, random_state=42, max_folds=MAX_SELECTION_FOLDS)
final_plan, final_n_folds = build_pump_combination_plan(
    single_id_subset, random_state=42, max_folds=None)
print('Preliminary evaluation folds:', selection_n_folds)
print('Final exhaustive folds:', final_n_folds)

In [ ]:
# Collapse fault severity digits to 1, preserving the defect position.
def defect_pattern(fault_id):
    return ''.join('0' if c == '0' else '1' for c in fault_id.rstrip('_')) + '_'

patterns = sorted({defect_pattern(fault_id) for fault_id in filt_unique_ids}
                  - {healthy_label}, reverse=True)
class_label_meta = {
    pattern: (chr(ord('A') + i) if i < 26 else f'D{i + 1}')
    for i, pattern in enumerate(patterns)
}
class_label_meta[healthy_label] = '0'
complete_class_label_meta = {
    fault_id: class_label_meta[defect_pattern(fault_id)]
    for fault_id in filt_unique_ids
}
complete_class_label_meta

## 9. Per-model feature selection

Run deterministic exhaustive feature selection and evaluation separately for SVC, logistic regression, and decision tree models.

### SVC

In [ ]:
meta_scores_all_combs = {}
meta_cols_all_combs = {}

for comb in tqdm(all_combs):
    score, cols = feature_selection(
        single_id_subset,
        comb,
        model=MODEL_META["svc"],
        n_feats=60,
        random_state=42,
        max_folds=MAX_SELECTION_FOLDS
    )
    meta_scores_all_combs[comb] = float(score)
    meta_cols_all_combs[comb] = list(cols)

    del score, cols
    gc.collect()

In [ ]:
curr_test_ids_test = filt_unique_ids
subset = merged1[
    merged1["id"].isin(curr_test_ids_test)
].reset_index(drop=True)

true_labels, pred_labels = algo_test(
    subset,
    model=MODEL_META["svc"],
    cols_meta=meta_cols_all_combs,
    random_state=42,
    max_folds=MAX_SELECTION_FOLDS
)

### Logistic regression

In [ ]:
meta_scores_all_combs_lr = {}
meta_cols_all_combs_lr = {}

for comb in tqdm(all_combs):
    score, cols = feature_selection(
        single_id_subset,
        comb,
        model=MODEL_META["lr"],
        n_feats=60,
        random_state=42,
        max_folds=MAX_SELECTION_FOLDS
    )
    meta_scores_all_combs_lr[comb] = float(score)
    meta_cols_all_combs_lr[comb] = list(cols)

    del score, cols
    gc.collect()

In [ ]:
curr_test_ids_test = filt_unique_ids
subset = merged1[
    merged1["id"].isin(curr_test_ids_test)
].reset_index(drop=True)

true_labels, pred_labels = algo_test(
    subset,
    model=MODEL_META["lr"],
    cols_meta=meta_cols_all_combs_lr,
    random_state=42,
    max_folds=MAX_SELECTION_FOLDS
)

### Decision tree

In [ ]:
meta_scores_all_combs_dt = {}
meta_cols_all_combs_dt = {}

for comb in tqdm(all_combs):
    score, cols = feature_selection(
        single_id_subset,
        comb,
        model=MODEL_META["dt"],
        n_feats=60,
        random_state=42,
        max_folds=MAX_SELECTION_FOLDS
    )
    meta_scores_all_combs_dt[comb] = float(score)
    meta_cols_all_combs_dt[comb] = list(cols)

    del score, cols
    gc.collect()

In [ ]:
curr_test_ids_test = filt_unique_ids
subset = merged1[
    merged1["id"].isin(curr_test_ids_test)
].reset_index(drop=True)

true_labels, pred_labels = algo_test(
    subset,
    model=MODEL_META["dt"],
    cols_meta=meta_cols_all_combs_dt,
    random_state=42,
    max_folds=MAX_SELECTION_FOLDS
)

## 10. Combined model selection and ensemble

Choose the strongest model for each fault pair, assemble its selected features, and define the ensemble transformer.

In [ ]:
expected_pairs = set(all_combs)

for name, scores, cols in [
    ("SVC", meta_scores_all_combs, meta_cols_all_combs),
    ("LR", meta_scores_all_combs_lr, meta_cols_all_combs_lr),
    ("DT", meta_scores_all_combs_dt, meta_cols_all_combs_dt)
]:
    missing_scores = expected_pairs - set(scores)
    missing_cols = expected_pairs - set(cols)
    if missing_scores or missing_cols:
        raise ValueError(
            f"{name} incomplete. "
            f"Missing scores={len(missing_scores)}, "
            f"missing columns={len(missing_cols)}"
        )

models_efficiency_list = {}
meta_combined_model_cols = {}
meta_combined_model_scores = {}

for key in all_combs:
    scores = {
        "lr": float(meta_scores_all_combs_lr[key]),
        "svc": float(meta_scores_all_combs[key]),
        "dt": float(meta_scores_all_combs_dt[key])
    }
    best_model = max(scores, key=scores.get)

    models_efficiency_list[key] = best_model
    if best_model == "lr":
        meta_combined_model_cols[key] = meta_cols_all_combs_lr[key]
    elif best_model == "svc":
        meta_combined_model_cols[key] = meta_cols_all_combs[key]
    else:
        meta_combined_model_cols[key] = meta_cols_all_combs_dt[key]

    meta_combined_model_scores[key] = scores[best_model]

print("Selected models:")
print(pd.Series(models_efficiency_list).value_counts())
print("Total pairs:", len(models_efficiency_list))

In [ ]:
class CombModelTransformer(BaseEstimator, TransformerMixin):

    def __init__(self, model=None, ids='', columns=None):
        self.model = model
        self.ids = ids
        self.columns = columns

    def fit(self, X, y=None):
        list_ids = self.ids.split('@')
        X = pd.DataFrame(X, columns=self.columns)
        X = X[X['id'].isin(list_ids)].reset_index(drop=True)
        y = X['id']
        X = X.drop(drop_cols, axis=1)
        self.scaler_ = RobustScaler()
        X = self.scaler_.fit_transform(X)
        self.model_ = copy.deepcopy(self.model)
        self.model_.fit(X, y)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns)
        X = X.drop(drop_cols, axis=1)
        X = self.scaler_.transform(X)
        return self.model_.predict(X).reshape(-1, 1)

In [ ]:
def comb_algo_test(df, cols_meta=None, random_state=42):
    model_meta = MODEL_META
    true_labels = []
    pred_labels = []

    le = LabelEncoder()
    _ = le.fit_transform(df["id"])
    mapped_classes = {
        i: class_id for i, class_id in enumerate(le.classes_)
    }

    combined_process = FeatureUnion([
        (
            f"pipe_{i}",
            Pipeline([
                (
                    f"{key}ct",
                    ColumnTransformer(
                        transformers=[
                            ("cols", "passthrough", cols + drop_cols)
                        ]
                    )
                ),
                (
                    f"{key}model",
                    CombModelTransformer(
                        model=model_meta[models_efficiency_list[key]],
                        ids=key,
                        columns=cols + drop_cols
                    )
                )
            ])
        )
        for i, (key, cols) in enumerate(cols_meta.items())
    ])

    split_plan, n_folds = build_pump_combination_plan(
        df,
        random_state=random_state,
        max_folds=None
    )
    print("Exhaustive pump-combination folds:", n_folds)

    for fold in tqdm(range(n_folds), desc="Final exhaustive folds"):
        train_data, test_data = train_test_split_2(
            df,
            fold=fold,
            split_plan=split_plan,
            n_folds=n_folds
        )

        iter_combined_process = copy.deepcopy(combined_process)
        iter_combined_process.fit(train_data)
        output = iter_combined_process.transform(test_data)

        pred_labels.extend([mode(arr) for arr in output])
        true_labels.extend(test_data["id"].tolist())

        del iter_combined_process, output
        del train_data, test_data

        if (fold + 1) % 10 == 0:
            gc.collect()

    gc.collect()

    plt.figure(figsize=(28, 18))
    conf_matr = confusion_matrix(true_labels, pred_labels)
    arr_true_labels = np.array(true_labels)
    sample_weights = [
        arr_true_labels[arr_true_labels == class_id].shape[0]
        for class_id in sorted(np.unique(arr_true_labels))
    ]
    normalized_conf_matr = np.round(
        np.array([
            row / sample_weights[i]
            for i, row in enumerate(conf_matr)
        ]),
        3
    )
    sns.heatmap(normalized_conf_matr, annot=True)

    print(mapped_classes)
    print(classification_report(true_labels, pred_labels, zero_division=0))
    print(
        "Macro F1:",
        f1_score(true_labels, pred_labels, average="macro", zero_division=0)
    )
    print(
        "Weighted F1:",
        f1_score(true_labels, pred_labels, average="weighted", zero_division=0)
    )

    return true_labels, pred_labels

## 11. Combined evaluation and defect-level reporting

Evaluate the selected-model ensemble over every per-fault test-pump combination and report normalized confusion matrices and F1 scores at both fault-ID and defect-label levels. This is intentionally the longest evaluation stage.

### Ensemble evaluation

In [ ]:
true_labels, pred_labels = comb_algo_test(
    subset,
    cols_meta=meta_combined_model_cols,
    random_state=42
)

### Defect-label evaluation

In [ ]:
true_defects = [complete_class_label_meta[label] for label in true_labels]
pred_defects = [complete_class_label_meta[label] for label in pred_labels]
plt.figure(figsize=(16, 12))
conf_matr = confusion_matrix(true_defects, pred_defects)
arr_true_defects = np.array(true_defects)
sample_weights = [arr_true_defects[arr_true_defects == clsid].shape[0] for clsid in sorted(np.unique(arr_true_defects))]
updt_conf_matr = np.matrix.round(np.array([row / sample_weights[i] for i, row in enumerate(conf_matr)]), 3)
sns.heatmap(updt_conf_matr, annot=True)
print(classification_report(true_defects, pred_defects))
print(f1_score(true_defects, pred_defects, average='weighted'))
print('Macro F1:', f1_score(true_defects, pred_defects, average='macro', zero_division=0))
print('Weighted F1:', f1_score(true_defects, pred_defects, average='weighted', zero_division=0))

In [ ]:
# PUMP legend: use actual fault-code patterns rather than FAN defect names.
class_legend = pd.DataFrame([
    {'label': label, 'fault_pattern': pattern,
     'fault_ids': ', '.join(fault_id for fault_id in filt_unique_ids
                           if defect_pattern(fault_id) == pattern)}
    for pattern, label in class_label_meta.items()
]).sort_values('label').reset_index(drop=True)
print(class_legend.to_string(index=False))

In [ ]:
# healthy_label was derived from the selected PUMP classes above.

true_health = ['Healthy' if label == healthy_label else 'Non-Healthy' for label in true_labels]

pred_health = ['Healthy' if label == healthy_label else 'Non-Healthy' for label in pred_labels]

plt.figure(figsize=(8, 6))

conf_matr = confusion_matrix(true_health, pred_health)

arr_true_health = np.array(true_health)

sample_weights = [arr_true_health[arr_true_health == clsid].shape[0]
                  for clsid in sorted(np.unique(arr_true_health))]

updt_conf_matr = np.matrix.round(
    np.array([row / sample_weights[i] for i, row in enumerate(conf_matr)]),
    3
)

sns.heatmap(
    updt_conf_matr,
    annot=True,
    xticklabels=sorted(np.unique(arr_true_health)),
    yticklabels=sorted(np.unique(arr_true_health))
)

print(classification_report(true_health, pred_health))

print(f1_score(true_health, pred_health, average='weighted'))

print('Macro F1:',
      f1_score(true_health, pred_health, average='macro', zero_division=0))

print('Weighted F1:',
      f1_score(true_health, pred_health, average='weighted', zero_division=0))

print(f"Healthy: {healthy_label}; Non-Healthy: all remaining PUMP fault IDs")

In [ ]:
# Select which final exhaustive fold you want to inspect.
# Valid values: 0 to audit_n_folds - 1
AUDIT_FOLD = 0

audit_plan, audit_n_folds = build_pump_combination_plan(
    subset,
    random_state=42,
    max_folds=None  # Complete exhaustive final-evaluation plan
)

if not 0 <= AUDIT_FOLD < audit_n_folds:
    raise ValueError(
        f"AUDIT_FOLD must be between 0 and {audit_n_folds - 1}"
    )

audit_train, audit_test = train_test_split_2(
    subset,
    fold=AUDIT_FOLD,
    split_plan=audit_plan,
    n_folds=audit_n_folds
)

audit_rows = []

for fault_id in sorted(subset["id"].unique()):
    testing_pumps = sorted(
        audit_test.loc[
            audit_test["id"] == fault_id,
            "pump_no"
        ].astype(int).unique().tolist()
    )

    training_pumps = sorted(
        audit_train.loc[
            audit_train["id"] == fault_id,
            "pump_no"
        ].astype(int).unique().tolist()
    )

    audit_rows.append({
        "fault_id": fault_id,
        "testing_pumps": testing_pumps,
        "training_pumps": training_pumps,
        "total_test_combinations": len(audit_plan[fault_id])
    })

pump_split_audit = pd.DataFrame(audit_rows)

print(
    f"Final evaluation has {audit_n_folds} folds. "
    f"Showing fold {AUDIT_FOLD}."
)
print(
    "Because validation is exhaustive, there is no permanent split: "
    "every pump enters testing in some folds and training in other folds."
)

pump_split_audit

## 12. Export artifacts and audit merged coverage

Save selected features/model choices and verify expected samples are represented in the merged dataset.

### Export model metadata and merged datasets

In [ ]:
with open(OUTPUT_DIR / 'combwise_cols_lr_svc_dt_clean_exhaustive_pumps.json', 'w') as f:
    json.dump(meta_combined_model_cols, f)
with open(OUTPUT_DIR / 'combwise_scores_lr_svc_dt_clean_exhaustive_pumps.json', 'w') as f:
    json.dump(meta_combined_model_scores, f)
with open(OUTPUT_DIR / 'combwise_model_selection_lr_svc_dt_clean_exhaustive_pumps.json', 'w') as f:
    json.dump(models_efficiency_list, f)
merged.to_csv(OUTPUT_DIR / 'merged_clean_exhaustive_pumps_lr_svc_dt.csv', index=False)
print('Final merged samples:', merged['sample_id_key'].nunique())
print('\nSamples per fault:')
print(merged.groupby('id').size())
if merged['sample_id_key'].duplicated().any():
    raise ValueError('Duplicate sample_id_key found in merged data')
merged1.to_csv(OUTPUT_DIR / 'merged_clean_model_input_exhaustive_pumps_lr_svc_dt.csv', index=False)
with open(OUTPUT_DIR / 'PUMP_class_label_mapping.json', 'w') as f:
    json.dump(complete_class_label_meta, f, indent=2)
pump_split_audit.to_csv(OUTPUT_DIR / 'PUMP_train_test_pump_audit.csv', index=False)
class_legend.to_csv(OUTPUT_DIR / 'PUMP_class_legend.csv', index=False)
print('Saved outputs to:', OUTPUT_DIR.resolve())

### Coverage audit

In [ ]:
# Audit all four inputs; matching keys were enforced before merging.
print(input_coverage.to_string(index=False))
print('Samples retained after merge:', merged['sample_id_key'].nunique())
print('Samples used for healthy/single-fault selection:', single_id_subset['sample_id_key'].nunique())
coverage = merged.groupby(['id', 'pump_no']).agg(
    included_samples=('sample_id_key', 'nunique')
).reset_index()
coverage.to_csv(OUTPUT_DIR / 'PUMP_merged_sample_coverage.csv', index=False)
input_coverage.to_csv(OUTPUT_DIR / 'PUMP_input_file_coverage.csv', index=False)
coverage